# S4D vs. Transformer: CUDA run

Runs the existing context-length sweep on an NVIDIA GPU with `--device cuda`. Model, optimizer,
sequence lengths, batch size, step counts and seeds are exactly those in
`experiments/run_context_length_sweep.py`.

The results form a new hardware dataset under `results/cuda/`. The historical Apple MPS results
in `results/*.json` are not touched. This notebook reports numbers and does not declare a faster
device or a better model. See `REMOTE_GPU.md` for the timing and memory methodology.

Runtime: **Runtime > Change runtime type > GPU**.

## 1. GPU

In [1]:
!nvidia-smi

Sat Oct  3 00:08:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Configuration

Set `REF` to the commit SHA (or tag) you intend to measure. A branch name works but is less reproducible.

In [2]:
REPO_URL = "https://github.com/AdebanjiAdelowo/s4d-transformer-comparison.git"
REF = "117ab7957c0a7e48a9d9070012f5bb3827827738"            # e.g. a full commit SHA; required
WORKDIR = "/content/s4d-transformer-comparison"
SWEEP_DIR = "results/cuda"
SMOKE_DIR = "results/cuda/smoke"   # inside results/cuda; sweep and plots only read results/cuda/*_bs*.json

assert REF, "Set REF to the commit SHA to measure before running the rest of the notebook."

## 3. Clone and check out `REF`

In [3]:
import json, os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not os.path.isdir(WORKDIR):
    sh(f"git clone {REPO_URL} {WORKDIR}")
sh("git fetch --all --tags", cwd=WORKDIR)
sh(f"git checkout --detach {REF}", cwd=WORKDIR)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("checked out", COMMIT)
assert os.path.exists("src/device.py"), "REF predates --device support; choose a newer commit."

$ git clone https://github.com/AdebanjiAdelowo/s4d-transformer-comparison.git /content/s4d-transformer-comparison
$ git fetch --all --tags
$ git checkout --detach 117ab7957c0a7e48a9d9070012f5bb3827827738
checked out 117ab7957c0a7e48a9d9070012f5bb3827827738


## 4. PyTorch before dependency installation

In [4]:
import platform, torch
TORCH_BEFORE = {"torch": torch.__version__, "cuda": torch.version.cuda, "cuda_available": torch.cuda.is_available()}
print("python", platform.python_version(), "| before install:", TORCH_BEFORE)

python 3.13.15 | before install: {'torch': '2.11.0+cu130', 'cuda': '13.0', 'cuda_available': True}


## 5. Install dependencies

Colab ships a CUDA build of PyTorch that satisfies `torch>=2.2`, so pip should leave it in place. The next cell checks that.

In [5]:
sh(f"{sys.executable} -m pip install -q -r requirements.txt")
TORCH_AFTER = json.loads(subprocess.check_output([sys.executable, "-c",
    "import json, torch; print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, "
    "'cuda_available': torch.cuda.is_available()}))"], text=True))
print("before install:", TORCH_BEFORE)
print("after install: ", TORCH_AFTER)
assert TORCH_AFTER["cuda_available"], "pip replaced or broke the CUDA build of PyTorch; do not continue."
if TORCH_AFTER["torch"] != TORCH_BEFORE["torch"]:
    print("WARNING: torch version changed during install; restart the runtime before continuing.")

$ /usr/bin/python3 -m pip install -q -r requirements.txt
before install: {'torch': '2.11.0+cu130', 'cuda': '13.0', 'cuda_available': True}
after install:  {'torch': '2.11.0+cu130', 'cuda': '13.0', 'cuda_available': True}


## 6. Environment report (after install)

In [6]:
print("python ", platform.python_version())
print("torch  ", torch.__version__)
print("cuda   ", torch.version.cuda, "| available:", torch.cuda.is_available())
print("cudnn  ", torch.backends.cudnn.version())
assert torch.cuda.is_available(), "CUDA is not available after install; do not continue."
GPU_NAME = torch.cuda.get_device_name(0)
print("gpu    ", GPU_NAME)

python  3.13.15
torch   2.11.0+cu130
cuda    13.0 | available: True
cudnn   92700
gpu     Tesla T4


## 7. Tests

The numerical tests (convolution vs. recurrence, stability constraint, backbone checks) and the device/output tests.

In [7]:
sh(f"{sys.executable} -m pytest tests/ -v")

$ /usr/bin/python3 -m pytest tests/ -v


## 8. Short CUDA smoke runs

60 steps at block_size=256 for each mixer, same model width/depth as the sweep. These are a pipeline check, not results.

In [8]:
shutil.rmtree(SMOKE_DIR, ignore_errors=True)   # never let a stale smoke file pass the gate
SMOKE_ARGS = ("--block_size 256 --max_iters 60 --eval_interval 30 --eval_iters 5 "
              "--n_layer 4 --n_embd 128 --n_head 4 --s4d_state 64 --batch_size 32 --seed 42 "
              "--tag cuda_smoke --device cuda --overwrite")
sh(f"{sys.executable} src/train.py --mixer s4d --out {SMOKE_DIR}/s4d_bs256.json {SMOKE_ARGS}")

$ /usr/bin/python3 src/train.py --mixer s4d --out results/cuda/smoke/s4d_bs256.json --block_size 256 --max_iters 60 --eval_interval 30 --eval_iters 5 --n_layer 4 --n_embd 128 --n_head 4 --s4d_state 64 --batch_size 32 --seed 42 --tag cuda_smoke --device cuda --overwrite


In [9]:
sh(f"{sys.executable} src/train.py --mixer attn --out {SMOKE_DIR}/attn_bs256.json {SMOKE_ARGS}")

$ /usr/bin/python3 src/train.py --mixer attn --out results/cuda/smoke/attn_bs256.json --block_size 256 --max_iters 60 --eval_interval 30 --eval_iters 5 --n_layer 4 --n_embd 128 --n_head 4 --s4d_state 64 --batch_size 32 --seed 42 --tag cuda_smoke --device cuda --overwrite


## 9. Check smoke metrics and CUDA metadata

In [10]:
import math

def _pos(x):
    return isinstance(x, (int, float)) and math.isfinite(x) and x > 0

def check_cuda_result(path):
    r = json.load(open(path))
    vals = r["loss_curve"]["train_loss"] + r["loss_curve"]["val_loss"]
    assert all(math.isfinite(v) for v in vals), f"non-finite loss in {path}"
    assert r["device"] == "cuda", f"{path} ran on {r['device']}"
    assert r["device_info"]["requested"] == "cuda"
    assert r["device_info"]["gpu_name"], "missing GPU name"
    m = r["memory"]
    assert m["cuda_max_memory_allocated_bytes"] and m["cuda_max_memory_reserved_bytes"]
    assert m["mps_current_allocated_bytes_sampled_max"] is None
    assert m["device_memory_api"] and m["device_memory_api"].startswith("torch.cuda.max_memory_allocated")
    t = r["timing"]
    for k in ("total_seconds", "mean_step_seconds", "median_step_seconds", "eval_seconds",
              "tokens_per_second", "train_step_tokens_per_second"):
        assert _pos(t[k]), f"{path}: timing field {k} = {t[k]!r}"
    assert "definitions" in t
    assert math.isfinite(r["final_val_loss"]) and math.isfinite(r["final_val_perplexity"])
    return r

def smoke_passed():
    for mixer in ("s4d", "attn"):
        r = check_cuda_result(f"{SMOKE_DIR}/{mixer}_bs256.json")
        assert r["tag"] == "cuda_smoke" and r["mixer"] == mixer

smoke_passed()
smoke = {m: check_cuda_result(f"{SMOKE_DIR}/{m}_bs256.json") for m in ("s4d", "attn")}
for m, r in smoke.items():
    t, mem = r["timing"], r["memory"]
    print(f"{m:4s} final val {r['final_val_loss']:.4f} | total {t['total_seconds']:.1f}s | "
          f"median step {t['median_step_seconds']*1e3:.2f} ms | "
          f"peak alloc {mem['cuda_max_memory_allocated_bytes']/2**20:.1f} MiB | {r['device_info']['gpu_name']}")
print("smoke checks passed")

s4d  final val 3.1077 | total 3.7s | median step 23.78 ms | peak alloc 359.4 MiB | Tesla T4
attn final val 2.7308 | total 3.3s | median step 38.17 ms | peak alloc 615.0 MiB | Tesla T4
smoke checks passed


## 10. Gate

Review the smoke output above. The gate re-reads both smoke JSON files and re-runs every check. The full sweep is 10 runs (5 context lengths x 2 mixers). On the
historical Apple MPS machine it took about 18 minutes in total. CUDA time on this GPU has not
been measured, so check the smoke timings before you commit to it. Set `RUN_FULL_SWEEP = True`
to continue.

In [12]:
RUN_FULL_SWEEP = True
OVERWRITE = False   # True only to replace an earlier CUDA sweep in this runtime

smoke_passed()   # raises unless both smoke runs exist and pass every check
if not RUN_FULL_SWEEP:
    raise SystemExit("Gate closed: set RUN_FULL_SWEEP = True and re-run this cell to continue.")

## 11. Full context-length sweep on CUDA

The unchanged sweep from `experiments/run_context_length_sweep.py`, with outputs in `results/cuda/`.

In [13]:
assert RUN_FULL_SWEEP, "gate closed"
smoke_passed()
assert os.path.normpath(SWEEP_DIR) == os.path.join("results", "cuda")
sh(f"{sys.executable} experiments/run_context_length_sweep.py --device cuda --out_dir {SWEEP_DIR}"
   + (" --overwrite" if OVERWRITE else ""))

$ /usr/bin/python3 experiments/run_context_length_sweep.py --device cuda --out_dir results/cuda


## 12. Integrity of the historical MPS results

In [14]:
sh("git diff --exit-code HEAD -- results/*.json results/summary_table.md figures/")
print("historical MPS results and figures unchanged")

$ git diff --exit-code HEAD -- results/*.json results/summary_table.md figures/
historical MPS results and figures unchanged


## 13. Summary (CUDA only)

- `tokens/s (incl. eval)` uses the same definition as the historical files: it includes evaluation, CPU batch assembly and host-to-device copies.
- `step tokens/s` covers only the synchronized optimizer steps.
- Peak memory is the exact CUDA allocator peak, and it is not comparable with the sampled MPS lower bound or with process RSS.

This table describes one GPU, one seed, and one run per configuration. It does not rank devices or models.

In [15]:
import glob
rows = sorted((check_cuda_result(f) for f in glob.glob(f"{SWEEP_DIR}/*_bs*.json")),
              key=lambda r: (r["config"]["block_size"], r["mixer"]))
assert len(rows) == 10, f"expected 10 sweep results, found {len(rows)}"
print(f"GPU: {rows[0]['device_info']['gpu_name']} | torch {rows[0]['device_info']['torch_version']} "
      f"| CUDA {rows[0]['device_info']['cuda_version']} | commit {COMMIT}\n")
hdr = ("mixer", "block", "iters", "val loss", "tok/s (incl. eval)", "step tok/s",
       "median step ms", "eval s", "peak alloc MiB", "peak reserved MiB")
print(" | ".join(hdr)); print(" | ".join("---" for _ in hdr))
for r in rows:
    t, m, c = r["timing"], r["memory"], r["config"]
    print(" | ".join(str(x) for x in (
        r["mixer"], c["block_size"], c["max_iters"], f"{r['final_val_loss']:.4f}",
        f"{t['tokens_per_second']:.0f}", f"{t['train_step_tokens_per_second']:.0f}",
        f"{t['median_step_seconds']*1e3:.2f}", f"{t['eval_seconds']:.1f}",
        f"{m['cuda_max_memory_allocated_bytes']/2**20:.1f}",
        f"{m['cuda_max_memory_reserved_bytes']/2**20:.1f}")))

GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0 | commit 117ab7957c0a7e48a9d9070012f5bb3827827738

mixer | block | iters | val loss | tok/s (incl. eval) | step tok/s | median step ms | eval s | peak alloc MiB | peak reserved MiB
--- | --- | --- | --- | --- | --- | --- | --- | --- | ---
attn | 64 | 1200 | 1.9501 | 137785 | 175362 | 10.42 | 2.5 | 121.4 | 138.0
s4d | 64 | 1200 | 1.7991 | 103839 | 127128 | 14.65 | 3.1 | 109.9 | 140.0
attn | 128 | 1200 | 1.9295 | 204813 | 259044 | 15.53 | 3.7 | 249.4 | 276.0
s4d | 128 | 1200 | 1.7278 | 199931 | 247456 | 15.14 | 3.5 | 194.0 | 220.0
attn | 256 | 1200 | 1.9200 | 161320 | 193626 | 42.10 | 8.8 | 615.0 | 722.0
s4d | 256 | 1200 | 1.6590 | 252186 | 308748 | 26.35 | 5.8 | 359.4 | 406.0
attn | 512 | 1200 | 1.9419 | 115774 | 136185 | 120.22 | 23.9 | 1860.0 | 1936.0
s4d | 512 | 1200 | 1.6066 | 272328 | 326700 | 49.88 | 10.5 | 692.3 | 742.0
attn | 1024 | 400 | 2.4645 | 67054 | 81480 | 403.22 | 34.0 | 6455.0 | 6892.0
s4d | 1024 | 400 | 1.9092 | 272579 | 3

## 14. Plots and summary table for `results/cuda`

Written into `results/cuda/`; the tracked `figures/` are not touched.

In [16]:
sh(f"{sys.executable} experiments/make_plots.py --results_dir {SWEEP_DIR}")
sh("git diff --exit-code HEAD -- results/*.json results/summary_table.md figures/")

$ /usr/bin/python3 experiments/make_plots.py --results_dir results/cuda
$ git diff --exit-code HEAD -- results/*.json results/summary_table.md figures/


## 15. Package CUDA outputs

In [17]:
import re, tempfile
env = {"commit": COMMIT, "ref": REF, "gpu_name": GPU_NAME, "torch_before_install": TORCH_BEFORE,
       "torch_after_install": TORCH_AFTER, "torch": torch.__version__,
       "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
       "python": platform.python_version(),
       "nvidia_smi": subprocess.check_output(["nvidia-smi"], text=True)}
stage = tempfile.mkdtemp()
shutil.copytree(SWEEP_DIR, os.path.join(stage, SWEEP_DIR))   # sweep JSON, plots, table, smoke/
json.dump(env, open(os.path.join(stage, "environment.json"), "w"), indent=2)
name = f"cuda_results_{re.sub(r'[^A-Za-z0-9]+', '_', GPU_NAME)}_{COMMIT[:8]}"
archive = shutil.make_archive(os.path.join(os.path.dirname(WORKDIR), name), "zip", stage)
print("wrote", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not on Colab: download the archive from the file browser (Kaggle: /kaggle/working).")

wrote /content/cuda_results_Tesla_T4_117ab795.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>